# YouTube Trending Video Analysis - Key Insights & Summary

This notebook consolidates empirical findings, key statistical takeaways, and strategic recommendations from our end-to-end exploratory analysis of over 105,000 trending YouTube videos across the United States (US), Great Britain (GB), and India (IN). It concludes with a curated showcase of the three most impactful visualizations generated during the study.

## Key Findings

Based on the exploratory data analysis and text mining conducted in notebooks `01_eda_overview` and `02_text_analysis`, the following trends emerge:

- **Category Performance (Music vs. Others)**:
  - **Music** stands out as the single highest-performing category by a wide margin, averaging **9.15M views per trending video**.
  - This is almost three times higher than the second-place category, **Movies** (average **3.19M views**), followed by **Nonprofits & Activism** (**2.96M views**), and general **Entertainment** (**1.64M views**).

- **Best Publishing Windows (Hour & Day of Week)**:
  - **Peak Day**: **Friday** produces the highest average viewership by far at **4.58M views**, followed by **Thursday** (**3.31M views**). Weekends experience lower initial trending views (Saturday: **1.78M**, Sunday: **2.82M**), confirming that Friday releases benefit most from weekend viewing momentum.
  - **Peak Hours**: Videos published during off-peak hours like **04:00 UTC** achieve the highest average view velocity (**6.73M views**), while creator upload volume peaks heavily during afternoon windows between **14:00 and 17:00 UTC** (over **8,000 videos published per hour**).

- **Cross-Country Comparison (Reach vs. Engagement Discrepancy)**:
  - **Great Britain (GB)** records the highest average view count per trending video at **5.79M views**, yet ranks second in engagement rate (**3.84%**).
  - The **United States (US)** achieves an average of **2.36M views** while delivering the highest overall engagement rate (**4.05%**).
  - **India (IN)** shows massive video volume but lower average views per trending entry (**0.79M views / 792K**) and an engagement rate of **2.55%**.

- **Regional Category Preferences**:
  - In **India**, **Entertainment** dominates with **14,297 trending videos** (~44.8% share), followed by **News & Politics** (**4,645 videos**).
  - In **Great Britain**, **Music** is the undisputed leader with **11,205 videos** (~34.6% share), surpassing Entertainment (**7,764 videos**).
  - In the **United States**, content is more distributed: **Entertainment** leads (**9,943 videos**), followed by **Music** (**6,467 videos**), and practical content like **Howto & Style** (**4,142 videos**).

- **Title Formatting Dynamics (Caps, Punctuation & Length)**:
  - **ALL-CAPS Penalty**: Titles featuring fully uppercase words average **2.35M views** compared to **3.22M views** for titles without ALL-CAPS words—a **27.0% lower average viewership**.
  - **Exclamation Mark Penalty**: Titles with exclamation marks (`!`) average **1.40M views**, representing a dramatic **54.6% drop** compared to titles without exclamation marks (**3.09M views**).
  - **Title Length Correlation**: Title character length exhibits a weakly negative Pearson correlation with views (**-0.0782**), indicating concise, clear titles outperform excessively long clickbait descriptions.

## Business Recommendations

For content creators, brands, and digital marketing teams aiming to maximize virality and audience engagement on YouTube:

1. **Align Release Timing with the Weekend Viewing Curve**:
   - Schedule flagship releases on **Thursdays or Fridays** (specifically late mornings to early afternoons UTC). This positions videos to capture the immediate end-of-week surge where average views peak above 4.5M.

2. **Adopt Region-Specific Content Strategies**:
   - **UK/Europe Market**: Focus resources on high-production Music, audio-visual collaborations, and lifestyle entertainment.
   - **US Market**: Prioritize community-building and interactive formats that invite likes and comments, capitalizing on the highest engagement rate (4.05%).
   - **India/South Asia Market**: Cater to high-volume episodic Entertainment, regional cinema releases, and current affairs commentary.

3. **Eliminate Aggressive Clickbait Conventions**:
   - Refrain from ALL-CAPS words and exclamation marks in titles. Modern recommendation algorithms and viewers demonstrate clear preference/trust toward clean, descriptive titles over sensory fatigue tactics.

4. **Target Music and High-Replay Formats for Exponential Reach**:
   - Where feasible, integrate musical elements, signature sound bites, or soundtrack collaborations. Music's 9.15M average view benchmark highlights the immense power of repeat viewership.

5. **Optimize Title Precision (40-60 Characters)**:
   - Keep titles concise and front-load key entities (creator name, franchise, primary topic) rather than filler phrases. Clean titles maintain strong CTR on mobile screens without truncation.

In [ ]:
# Highlights Section: Load data and re-generate the 3 most impactful charts
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup aesthetics and output directory
sns.set_theme(style="whitegrid")
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Load preprocessed dataset
data_path = Path("../data/processed_youtube_data.csv")
df = pd.read_csv(data_path, parse_dates=["trending_date", "publish_time"])
df["engagement_rate"] = (df["likes"] + df["dislikes"] + df["comment_count"]) / df["views"]
print(f"Loaded {len(df):,} records for summary visualization.\n")

# ======================================================================
# CHART 1: Top 10 Categories by Average Views
# ======================================================================
cat_summary = (
    df.groupby("category_name")["views"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .sort_values(ascending=True)
)

plt.figure(figsize=(10, 6))
palette = sns.color_palette("Blues_r", n_colors=len(cat_summary))
bars1 = plt.barh(
    cat_summary.index,
    cat_summary.values / 1e6,
    color=palette[::-1],
    height=0.65,
)

max_v1 = (cat_summary.values / 1e6).max()
for bar in bars1:
    w = bar.get_width()
    plt.text(
        w + (max_v1 * 0.015),
        bar.get_y() + bar.get_height() / 2,
        f"{w:.2f}M",
        va="center",
        fontsize=10,
        fontweight="semibold",
        color="#2c3e50",
    )

plt.title("Impact Highlight 1: Top 10 Categories by Average Views", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Average Views (Millions)", fontsize=12, labelpad=10)
plt.ylabel("Category", fontsize=12, labelpad=10)
plt.xlim(0, max_v1 * 1.15)
plt.tight_layout()

chart1_path = output_dir / "top_categories_views.png"
plt.savefig(chart1_path, dpi=300)
print(f"Saved Chart 1 to: {chart1_path}")
plt.show()

# ======================================================================
# CHART 2: Cross-Country Audience Reach vs. Engagement Rate (Dual-Axis)
# ======================================================================
country_summary = (
    df.groupby("country")
    .agg(
        mean_views=("views", "mean"),
        mean_engagement=("engagement_rate", "mean"),
    )
    .reset_index()
)

x = np.arange(len(country_summary))
width = 0.35

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

bars_views = ax1.bar(
    x - width / 2,
    country_summary["mean_views"] / 1e6,
    width=width,
    color="#2b5c8f",
    label="Mean Views (Millions)",
)
bars_eng = ax2.bar(
    x + width / 2,
    country_summary["mean_engagement"] * 100,
    width=width,
    color="#e67e22",
    label="Mean Engagement Rate (%)",
)

ax1.set_title("Impact Highlight 2: Cross-Country Audience Reach vs. Engagement", fontsize=14, fontweight="bold", pad=15)
ax1.set_xlabel("Country", fontsize=12, labelpad=10)
ax1.set_ylabel("Mean Views (Millions)", color="#2b5c8f", fontsize=12, labelpad=8)
ax2.set_ylabel("Mean Engagement Rate (%)", color="#e67e22", fontsize=12, labelpad=8)
ax1.set_xticks(x)
ax1.set_xticklabels(country_summary["country"], fontsize=11, fontweight="bold")
ax1.grid(True, linestyle="--", alpha=0.5)
ax2.grid(False)

for bar in bars_views:
    ax1.annotate(
        f"{bar.get_height():.2f}M",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontweight="semibold",
        color="#2b5c8f",
    )
for bar in bars_eng:
    ax2.annotate(
        f"{bar.get_height():.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontweight="semibold",
        color="#e67e22",
    )

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon=True)
plt.tight_layout()

chart2_path = output_dir / "country_views_vs_engagement.png"
plt.savefig(chart2_path, dpi=300)
print(f"Saved Chart 2 to: {chart2_path}")
plt.show()

# ======================================================================
# CHART 3: Weekly Viewership Cycle (Peak Day Highlighted)
# ======================================================================
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_summary = (
    df.groupby("publish_day_of_week")["views"]
    .mean()
    .reindex(days_order)
)
max_day = dow_summary.idxmax()
bar_colors = ["#e74c3c" if d == max_day else "#3498db" for d in dow_summary.index]

plt.figure(figsize=(10, 6))
bars3 = plt.bar(
    dow_summary.index,
    dow_summary.values / 1e6,
    color=bar_colors,
    width=0.6,
)

for bar in bars3:
    h = bar.get_height()
    plt.annotate(
        f"{h:.2f}M",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontweight="semibold",
    )

plt.title("Impact Highlight 3: Weekly Viewership Cycle (Friday Peak Highlighted)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Day of Week", fontsize=12, labelpad=10)
plt.ylabel("Average Views (Millions)", fontsize=12, labelpad=10)
plt.ylim(0, (dow_summary.max() / 1e6) * 1.15)
plt.tight_layout()

chart3_path = output_dir / "views_by_day_of_week.png"
plt.savefig(chart3_path, dpi=300)
print(f"Saved Chart 3 to: {chart3_path}")
plt.show()